# 챗봇들 간의 적대적인 대화

프롬프트가 다음과 같이 리스트로 구성된다는 것은 이미 익숙하실 겁니다:

```
[
    {"role": "system", "content": "여기에 시스템 메시지"},
    {"role": "user", "content": "여기에 사용자 프롬프트"}
]
```

사실 이 구조는 더 긴 대화 이력을 표현하는 데도 사용할 수 있습니다:

```
[
    {"role": "system", "content": "여기에 시스템 메시지"},
    {"role": "user", "content": "첫 번째 사용자 프롬프트"},
    {"role": "assistant", "content": "어시스턴트의 응답"},
    {"role": "user", "content": "새로운 사용자 프롬프트"},
]
```

그리고 이 방식을 사용하면 이력을 가진 더 긴 상호작용을 할 수 있습니다.

In [6]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

anthropic_url = "https://api.anthropic.com/v1/"

openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)


In [4]:
# GPT-4.1-mini와 Claude-haiku-4.5 사이의 대화를 만들어봅시다
# 저렴한 버전의 모델을 사용하므로 비용은 최소화됩니다

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"

gpt_system = "당신은 매우 논쟁적인 챗봇입니다; \
대화의 어떤 내용에도 동의하지 않고 모든 것에 시비를 걸며, 비꼬는 말투로 반박합니다."

claude_system = "당신은 매우 정중하고 예의 바른 챗봇입니다. 상대방이 하는 말에 \
최대한 동의하거나 공통점을 찾으려고 합니다. 상대방이 논쟁적으로 나오면, \
상대를 진정시키고 계속 대화를 이어가려고 노력합니다."

gpt_messages = ["안녕하세요"]
claude_messages = ["안녕하세요"]

In [8]:
messages = [{"role": "system", "content": gpt_system}]

for gpt, claude in zip(gpt_messages, claude_messages):
    messages.append({"role": "assistant", "content": gpt})
    messages.append({"role": "user", "content": claude})

messages

[{'role': 'system',
  'content': '당신은 매우 논쟁적인 챗봇입니다; 대화의 어떤 내용에도 동의하지 않고 모든 것에 시비를 걸며, 비꼬는 말투로 반박합니다.'},
 {'role': 'assistant', 'content': '안녕하세요'},
 {'role': 'user', 'content': '안녕하세요'}]

In [9]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_gpt()

In [ ]:
def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = anthropic.chat.completions.create(model=claude_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_claude()

In [ ]:
call_gpt()

In [ ]:
gpt_messages = ["안녕하세요"]
claude_messages = ["안녕하세요"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Claude:\n{claude_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)

    claude_next = call_claude()
    display(Markdown(f"### Claude:\n{claude_next}\n"))
    claude_messages.append(claude_next)